# Graphs

In [1]:
CLASSIFICATION_CSV_URL = (
    "https://raw.githubusercontent.com/disel-espol/"
    "hpc-and-edge-cloud-architectures/main/"
    "Journal_paper/classified_architectures_with_metadata.csv"
)

!git clone https://github.com/WiscADSL/Cloudscape.git

!wget -q -O /content/classified_architectures_with_metadata.csv "$CLASSIFICATION_CSV_URL"

CLASSIFICATION_CSV = "/content/classified_architectures_with_metadata.csv"

Cloning into 'Cloudscape'...
remote: Enumerating objects: 580, done.
remote: Counting objects: 100% (386/386), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 580 (delta 375), reused 359 (delta 359), pack-reused 194 (from 1)
Receiving objects: 100% (580/580), 1.24 MiB | 21.48 MiB/s, done.
Resolving deltas: 100% (415/415), done.


In [2]:
import os
import glob
import math
from collections import Counter

import pandas as pd
import networkx as nx


# =========================================================
# 1. CONFIGURATION
# =========================================================

GRAPH_DIR = "/content/Cloudscape/data/graphs"

# If you know exactly which attribute contains the service type,
# specify it here. For example:
#
# SERVICE_TYPE_ATTR = "service_type"
#
# If left as None, the code will attempt to detect it automatically.
SERVICE_TYPE_ATTR = None

# Possible names for the attribute representing the service type.
SERVICE_TYPE_CANDIDATES = [
    "service_type",
    "serviceType",
    "category",
    "type",
    "service",
    "label",
    "name"
]


# =========================================================
# 2. AUXILIARY FUNCTIONS
# =========================================================

def safe_value(value):
    """
    Converts an attribute value into a clean string.

    Returns None when the value is empty.
    """
    if value is None:
        return None

    value = str(value).strip()

    if value == "":
        return None

    return value


def detect_service_type_attribute(G):
    """
    Detects which node attribute may represent
    the service type.

    The candidate attribute with the highest coverage is selected,
    meaning the attribute that appears in the largest number of nodes.
    """
    if SERVICE_TYPE_ATTR is not None:
        return SERVICE_TYPE_ATTR

    coverage = {}

    for attribute in SERVICE_TYPE_CANDIDATES:
        count = sum(
            1
            for _, attrs in G.nodes(data=True)
            if safe_value(attrs.get(attribute)) is not None
        )

        coverage[attribute] = count

    # Select the attribute that appears in the largest number of nodes.
    best_attribute = max(coverage, key=coverage.get)

    # If no attribute was found.
    if coverage[best_attribute] == 0:
        return None

    return best_attribute


def calculate_entropy(values):
    """
    Computes Shannon entropy:

        H = -sum(p_i * log2(p_i))

    High entropy indicates a more diverse distribution
    of service types.

    Low entropy indicates that a small number of service types
    dominate the architecture.
    """
    clean_values = [
        safe_value(value)
        for value in values
        if safe_value(value) is not None
    ]

    if not clean_values:
        return 0.0

    counts = Counter(clean_values)
    total = len(clean_values)

    entropy = 0.0

    for count in counts.values():
        probability = count / total
        entropy -= probability * math.log2(probability)

    return entropy


def calculate_normalized_entropy(values):
    """
    Computes normalized entropy between 0 and 1.

    0:
        All nodes belong to the same service type.

    1:
        Service types are distributed as uniformly
        as possible.
    """
    clean_values = [
        safe_value(value)
        for value in values
        if safe_value(value) is not None
    ]

    if not clean_values:
        return 0.0

    number_of_types = len(set(clean_values))

    if number_of_types <= 1:
        return 0.0

    entropy = calculate_entropy(clean_values)
    maximum_entropy = math.log2(number_of_types)

    return entropy / maximum_entropy


def calculate_average_degree(G):
    """
    Computes the average degree.

    For directed graphs:
        total degree = in-degree + out-degree.

    For undirected graphs:
        standard node degree.
    """
    number_of_nodes = G.number_of_nodes()

    if number_of_nodes == 0:
        return 0.0

    degrees = [degree for _, degree in G.degree()]

    return sum(degrees) / number_of_nodes


def calculate_average_in_degree(G):
    """
    Computes the average in-degree.
    This metric is only meaningful for directed graphs.
    """
    if not G.is_directed() or G.number_of_nodes() == 0:
        return None

    return sum(
        degree for _, degree in G.in_degree()
    ) / G.number_of_nodes()


def calculate_average_out_degree(G):
    """
    Computes the average out-degree.
    This metric is only meaningful for directed graphs.
    """
    if not G.is_directed() or G.number_of_nodes() == 0:
        return None

    return sum(
        degree for _, degree in G.out_degree()
    ) / G.number_of_nodes()


def calculate_branching_metrics(G):
    """
    Computes workflow branching metrics.

    For a directed graph, a node is considered a branching node when:

        out_degree > 1

    Three metrics are returned:

    1. branching_nodes:
       Number of nodes that generate branches.

    2. additional_branches:
       Number of additional branches.

       For example, a node with out_degree = 3 generates:

           3 - 1 = 2 additional branches.

    3. maximum_branching_factor:
       Largest branching degree observed in the graph.
    """
    if G.number_of_nodes() == 0:
        return 0, 0, 0

    if G.is_directed():
        branching_degrees = [
            degree
            for _, degree in G.out_degree()
            if degree > 1
        ]
    else:
        # In an undirected graph, a node is considered a branching node
        # when it is connected to more than two nodes.
        branching_degrees = [
            degree
            for _, degree in G.degree()
            if degree > 2
        ]

    branching_nodes = len(branching_degrees)

    if G.is_directed():
        additional_branches = sum(
            degree - 1 for degree in branching_degrees
        )
    else:
        additional_branches = sum(
            degree - 2 for degree in branching_degrees
        )

    maximum_branching_factor = (
        max(branching_degrees)
        if branching_degrees
        else 0
    )

    return (
        branching_nodes,
        additional_branches,
        maximum_branching_factor
    )


def longest_path_in_dag(G):
    """
    Computes the longest path when the graph is a DAG.

    The path length is expressed as the number of edges.
    """
    if G.number_of_nodes() == 0:
        return 0

    return nx.dag_longest_path_length(G)


def calculate_maximum_path_length(G):
    """
    Computes the maximum path length.

    Cases:

    1. Directed acyclic graph:
       The longest path in the DAG is used.

    2. Directed graph with cycles:
       Strongly connected components are condensed.
       Each cycle becomes a single node, and the longest path
       is computed in the resulting condensed DAG.

    3. Undirected graph:
       The maximum diameter across connected components is computed.

    Returns:
        path length,
        method used,
        cycle indicator.
    """
    if G.number_of_nodes() == 0:
        return 0, "empty_graph", False

    # -----------------------------------------------------
    # Directed graph
    # -----------------------------------------------------
    if G.is_directed():

        # Convert MultiDiGraph to DiGraph to avoid issues.
        simple_G = nx.DiGraph(G)

        if nx.is_directed_acyclic_graph(simple_G):
            length = longest_path_in_dag(simple_G)

            return length, "dag_longest_path", False

        # The graph contains cycles.
        strongly_connected_components = list(
            nx.strongly_connected_components(simple_G)
        )

        condensed_G = nx.condensation(
            simple_G,
            strongly_connected_components
        )

        length = longest_path_in_dag(condensed_G)

        return length, "condensation_dag", True

    # -----------------------------------------------------
    # Undirected graph
    # -----------------------------------------------------
    simple_G = nx.Graph(G)

    components = [
        simple_G.subgraph(component).copy()
        for component in nx.connected_components(simple_G)
    ]

    if not components:
        return 0, "empty_graph", False

    maximum_diameter = max(
        nx.diameter(component)
        if component.number_of_nodes() > 1
        else 0
        for component in components
    )

    return maximum_diameter, "component_diameter", False


def calculate_components(G):
    """
    Returns the number of graph components.

    For directed graphs:
    - Weakly connected components.
    - Strongly connected components.

    For undirected graphs:
    - Connected components.
    """
    if G.number_of_nodes() == 0:
        return 0, 0

    if G.is_directed():
        weak_components = nx.number_weakly_connected_components(G)
        strong_components = nx.number_strongly_connected_components(G)

        return weak_components, strong_components

    connected_components = nx.number_connected_components(G)

    return connected_components, connected_components


def calculate_source_sink_nodes(G):
    """
    Computes source and sink nodes in the workflow.

    Source:
        in_degree = 0

    Sink:
        out_degree = 0
    """
    if not G.is_directed():
        return None, None

    source_nodes = sum(
        1
        for _, degree in G.in_degree()
        if degree == 0
    )

    sink_nodes = sum(
        1
        for _, degree in G.out_degree()
        if degree == 0
    )

    return source_nodes, sink_nodes


# =========================================================
# 3. MAIN FUNCTION FOR GRAPH ANALYSIS
# =========================================================

def analyze_graph(file_path):
    """
    Reads a GraphML file and returns its metrics
    as a dictionary.
    """
    G = nx.read_graphml(file_path)

    number_of_nodes = G.number_of_nodes()
    number_of_edges = G.number_of_edges()

    # -----------------------------------------------------
    # Degree metrics
    # -----------------------------------------------------
    average_degree = calculate_average_degree(G)
    average_in_degree = calculate_average_in_degree(G)
    average_out_degree = calculate_average_out_degree(G)

    # -----------------------------------------------------
    # Maximum path length
    # -----------------------------------------------------
    (
        maximum_path_length,
        path_method,
        contains_cycles
    ) = calculate_maximum_path_length(G)

    # -----------------------------------------------------
    # Branching metrics
    # -----------------------------------------------------
    (
        branching_nodes,
        additional_branches,
        maximum_branching_factor
    ) = calculate_branching_metrics(G)

    # -----------------------------------------------------
    # Service type and entropy
    # -----------------------------------------------------
    service_type_attribute = detect_service_type_attribute(G)

    if service_type_attribute is not None:
        service_types = [
            attrs.get(service_type_attribute)
            for _, attrs in G.nodes(data=True)
        ]

        valid_service_types = [
            safe_value(value)
            for value in service_types
            if safe_value(value) is not None
        ]

        service_type_entropy = calculate_entropy(valid_service_types)

        normalized_service_type_entropy = (
            calculate_normalized_entropy(valid_service_types)
        )

        number_of_service_types = len(set(valid_service_types))
        nodes_with_service_type = len(valid_service_types)

    else:
        service_type_entropy = 0.0
        normalized_service_type_entropy = 0.0
        number_of_service_types = 0
        nodes_with_service_type = 0

    # -----------------------------------------------------
    # Components
    # -----------------------------------------------------
    (
        weak_components,
        strong_components
    ) = calculate_components(G)

    # -----------------------------------------------------
    # Sources and sinks
    # -----------------------------------------------------
    source_nodes, sink_nodes = calculate_source_sink_nodes(G)

    # -----------------------------------------------------
    # Density
    # -----------------------------------------------------
    density = nx.density(G) if number_of_nodes > 1 else 0.0

    # -----------------------------------------------------
    # Results
    # -----------------------------------------------------
    return {
        "file": os.path.basename(file_path),
        "graph_type": type(G).__name__,
        "is_directed": G.is_directed(),

        "nodes": number_of_nodes,
        "edges": number_of_edges,
        "density": density,

        "average_degree": average_degree,
        "average_in_degree": average_in_degree,
        "average_out_degree": average_out_degree,

        "maximum_path_length": maximum_path_length,
        "maximum_path_method": path_method,
        "contains_cycles": contains_cycles,

        "branching_nodes": branching_nodes,
        "additional_branches": additional_branches,
        "maximum_branching_factor": maximum_branching_factor,

        "source_nodes": source_nodes,
        "sink_nodes": sink_nodes,

        "weak_components": weak_components,
        "strong_components": strong_components,

        "service_type_attribute": service_type_attribute,
        "nodes_with_service_type": nodes_with_service_type,
        "number_of_service_types": number_of_service_types,
        "service_type_entropy": service_type_entropy,
        "normalized_service_type_entropy": (
            normalized_service_type_entropy
        )
    }


# =========================================================
# 4. PROCESS ALL GRAPHML FILES
# =========================================================

files = sorted(
    glob.glob(
        os.path.join(GRAPH_DIR, "*.graphml")
    )
)

if not files:
    raise FileNotFoundError(
        f"No GraphML files were found in: {GRAPH_DIR}"
    )

results = []

for index, file_path in enumerate(files, start=1):
    try:
        metrics = analyze_graph(file_path)
        results.append(metrics)

    except Exception as error:
        print(
            f"Error processing "
            f"{os.path.basename(file_path)}: {error}"
        )

    if index % 50 == 0:
        print(f"Processed {index} of {len(files)} graphs")


# Create DataFrame
df_graph_metrics = pd.DataFrame(results)

# Sort by file name
df_graph_metrics = df_graph_metrics.sort_values(
    by="file"
).reset_index(drop=True)


# =========================================================
# 5. DISPLAY RESULTS
# =========================================================

print("\nNumber of graphs analyzed:", len(df_graph_metrics))

display(
    df_graph_metrics.head(10)
)


# =========================================================
# 6. SAVE RESULTS
# =========================================================

output_path = os.path.join(
    GRAPH_DIR,
    "graph_metrics.csv"
)

df_graph_metrics.to_csv(
    output_path,
    index=False
)

print("\nFile saved to:")
print(output_path)

Processed 50 of 396 graphs
Processed 100 of 396 graphs
Processed 150 of 396 graphs
Processed 200 of 396 graphs
Processed 250 of 396 graphs
Processed 300 of 396 graphs
Processed 350 of 396 graphs

Number of graphs analyzed: 396


,file,graph_type,is_directed,nodes,edges,density,average_degree,average_in_degree,average_out_degree,maximum_path_length,...,maximum_branching_factor,source_nodes,sink_nodes,weak_components,strong_components,service_type_attribute,nodes_with_service_type,number_of_service_types,service_type_entropy,normalized_service_type_entropy
0,-3lnf5lzsH0.graphml,DiGraph,True,13,16,0.102564,2.461538,1.230769,1.230769,1,...,3,6,0,1,7,service,13,12,3.546594,0.989297
1,-S-R7MWRpaI.graphml,MultiDiGraph,True,8,11,0.196429,2.750000,1.375000,1.375000,3,...,3,0,1,1,5,service,8,7,2.750000,0.979570
2,-ahWdCysMYw.graphml,DiGraph,True,8,9,0.160714,2.250000,1.125000,1.125000,3,...,2,2,1,1,7,service,8,8,3.000000,1.000000
3,-kA0ahrhX3I.graphml,DiGraph,True,9,8,0.111111,1.777778,0.888889,0.888889,2,...,3,3,4,3,7,service,9,7,2.725481,0.970836
4,-wLEkq21cvA.graphml,DiGraph,True,9,10,0.138889,2.222222,1.111111,1.111111,3,...,3,2,3,1,7,service,9,5,2.113283,0.910142
5,07lfvavMdfU.graphml,DiGraph,True,10,14,0.155556,2.800000,1.400000,1.400000,5,...,4,1,2,1,7,service,10,10,3.321928,1.000000
6,0F7KDLz-kIQ.graphml,MultiDiGraph,True,13,28,0.179487,4.307692,2.153846,2.153846,2,...,5,1,0,1,3,service,13,11,3.392747,0.980724
7,0JxJpNjI9Y0.graphml,MultiDiGraph,True,9,17,0.236111,3.777778,1.888889,1.888889,1,...,4,0,1,1,2,service,9,7,2.725481,0.970836
8,0gNMEyei-co.graphml,DiGraph,True,9,13,0.180556,2.888889,1.444444,1.444444,2,...,3,2,0,1,4,service,9,9,3.169925,1.000000
9,0wnNlOg42dc.graphml,DiGraph,True,8,12,0.214286,3.000000,1.500000,1.500000,2,...,2,0,0,1,3,service,8,8,3.000000,1.000000



File saved to:
/content/Cloudscape/data/graphs/graph_metrics.csv


In [3]:
# =========================================================
# GRAPH METRICS BY SELECTED ANALYTICAL CATEGORIES
# Run this AFTER the original GraphML metrics extraction cell
# =========================================================

import os
import ast
import re
import pandas as pd

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------
GRAPH_DIR = "/content/Cloudscape/data/graphs"

METRICS_PATH = os.path.join(
    GRAPH_DIR,
    "graph_metrics.csv"
)

CLASSIFICATION_CSV = (
    "/content/classified_architectures_with_metadata.csv"
)

# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------
def to_bool_series(series):
    """
    Converts a dataframe column to boolean safely.
    Supports bool values and CSV-like string values.
    """
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )


def parse_services(x):
    """
    Parses a service list stored as Python list, list-like string,
    or comma-separated text.
    """
    if isinstance(x, list):
        return [str(v).strip() for v in x]

    if pd.isna(x) or str(x).strip() == "":
        return []

    value = str(x).strip()

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed]

    except Exception:
        pass

    return [
        service.strip()
        for service in value.split(",")
        if service.strip()
    ]


def normalize_service_key(name):
    """
    Normalizes service names for robust matching.
    """
    value = str(name).strip()
    value = re.sub(r"\s+", " ", value)

    return re.sub(
        r"[^A-Za-z0-9]+",
        "",
        value
    ).lower()


def safe_filename(text):
    """
    Converts a category label into a safe filename fragment.
    """
    return re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        str(text)
    ).strip("_").lower()


def get_first_existing_flag(df, possible_columns, label):
    """
    Returns the first available boolean category column.
    """
    for column in possible_columns:
        if column in df.columns:
            return to_bool_series(df[column])

    raise KeyError(
        f"Could not find a valid flag for '{label}'. "
        f"Expected one of: {possible_columns}"
    )


def build_edge_no_cloudfront_flag(df):
    """
    Uses the persisted no-CloudFront flag if available.
    Otherwise, rebuilds it from the services column.
    """

    if "belongs_edge_no_cloudfront" in df.columns:
        return to_bool_series(df["belongs_edge_no_cloudfront"])

    if "tipo_arquitectura_no_cloudfront" in df.columns:
        return df["tipo_arquitectura_no_cloudfront"].isin(
            ["Edge", "Edge+HPC"]
        )

    if "services" not in df.columns:
        raise KeyError(
            "Cannot rebuild Edge no CloudFront because "
            "the 'services' column is unavailable."
        )

    print(
        "Warning: 'belongs_edge_no_cloudfront' was not found. "
        "It will be rebuilt from the service list."
    )

    edge_services = [
        "Snowball", "Snowcone", "Snowmobile", "SnowFamily",
        "SageMakerNeo", "SageMakerEdgeManager", "Monitron", "Panorama",
        "RoboMaker",
        "CloudFront",
        "Greengrass", "FreeRTOS", "IoTCore", "IoTSiteWise",
        "AlexaForBusiness",
        "LocalZones", "Wavelength", "Outpost",
        "StorageGateway",
        "UserConsumerCamera", "UserCompanyEdge", "UserConsumerEdge",
        "UserConsumerIOT", "UserConsumerPOS", "UserCompanyDrone",
        "UserConsumerFarmer",
        "UserConsumerAlexaGoogleHome",
        "UserCompanyElementalLiveDevice",
        "UserConsumerTV",
        "LambdaAtEdge", "IoT1Click"
    ]

    edge_services_no_cloudfront = [
        service
        for service in edge_services
        if normalize_service_key(service) != "cloudfront"
    ]

    edge_no_cloudfront_clean = {
        normalize_service_key(service)
        for service in edge_services_no_cloudfront
    }

    services_list = df["services"].apply(parse_services)

    return services_list.apply(
        lambda services: any(
            normalize_service_key(service) in edge_no_cloudfront_clean
            for service in services
        )
    )


# ---------------------------------------------------------
# 3. Load graph metrics
# ---------------------------------------------------------
if "df_graph_metrics" in globals():
    df_metrics = df_graph_metrics.copy()
    print("Using df_graph_metrics already available in memory.")

else:
    if not os.path.exists(METRICS_PATH):
        raise FileNotFoundError(
            f"Graph metrics file was not found: {METRICS_PATH}\n"
            "Run the original GraphML metrics cell first."
        )

    df_metrics = pd.read_csv(METRICS_PATH)
    print(f"Graph metrics loaded from: {METRICS_PATH}")

assert "file" in df_metrics.columns, (
    "Column 'file' was not found in graph metrics."
)

# Extract architecture ID from GraphML filename
df_metrics["architecture"] = (
    df_metrics["file"]
    .astype(str)
    .str.replace(r"\.graphml$", "", regex=True)
)

df_metrics["architecture"] = df_metrics["architecture"].astype(str)

print("Graphs in graph metrics:", len(df_metrics))
print(
    "Unique architecture IDs in graph metrics:",
    df_metrics["architecture"].nunique()
)

assert len(df_metrics) == df_metrics["architecture"].nunique(), (
    "Duplicate architecture IDs were found in graph metrics."
)

# ---------------------------------------------------------
# 4. Load classification dataframe
# ---------------------------------------------------------
if "df_final" in globals():
    df_categories = df_final.copy()
    print("Using df_final already available in memory.")

else:
    if not os.path.exists(CLASSIFICATION_CSV):
        raise FileNotFoundError(
            f"Classification CSV was not found: {CLASSIFICATION_CSV}"
        )

    df_categories = pd.read_csv(CLASSIFICATION_CSV)
    print(f"Classification dataset loaded from: {CLASSIFICATION_CSV}")

assert "architecture" in df_categories.columns, (
    "Column 'architecture' was not found in the classification dataset."
)

df_categories["architecture"] = (
    df_categories["architecture"]
    .astype(str)
)

df_categories = df_categories.drop_duplicates(
    subset="architecture"
).copy()

print("Architectures in classification dataset:", len(df_categories))

# ---------------------------------------------------------
# 5. Build selected category flags
# ---------------------------------------------------------

# Edge
if "belongs_edge" in df_categories.columns:
    edge_flag = to_bool_series(
        df_categories["belongs_edge"]
    )
else:
    edge_flag = df_categories["tipo_arquitectura"].isin(
        ["Edge", "Edge+HPC"]
    )

# HPC
if "belongs_hpc" in df_categories.columns:
    hpc_flag = to_bool_series(
        df_categories["belongs_hpc"]
    )
else:
    hpc_flag = df_categories["tipo_arquitectura"].isin(
        ["HPC", "Edge+HPC"]
    )

# Serverless
serverless_flag = get_first_existing_flag(
    df_categories,
    ["belongs_serverless", "has_serverless"],
    "Serverless"
)

# None
# This category represents architectures that do not belong
# to the main analytical Edge, HPC, or Serverless groups.
if "belongs_none_analytical" in df_categories.columns:
    none_flag = to_bool_series(
        df_categories["belongs_none_analytical"]
    )

elif "belongs_none" in df_categories.columns:
    none_flag = to_bool_series(
        df_categories["belongs_none"]
    )

else:
    print(
        "Warning: no persisted None-category flag was found. "
        "It will be derived as architectures that are not Edge, "
        "HPC, or Serverless."
    )

    none_flag = ~(
        edge_flag |
        hpc_flag |
        serverless_flag
    )


# Alternative Edge category excluding CloudFront
edge_no_cloudfront_flag = build_edge_no_cloudfront_flag(
    df_categories
)

# Serverless FaaS
faas_flag = get_first_existing_flag(
    df_categories,
    [
        "belongs_serverless_faas",
        "has_serverless_faas",
        "has_faas"
    ],
    "Serverless FaaS"
)

# Serverless Compute
serverless_compute_flag = get_first_existing_flag(
    df_categories,
    [
        "belongs_serverless_compute",
        "has_serverless_compute"
    ],
    "Serverless Compute"
)

# Serverless Event-Driven
event_driven_flag = get_first_existing_flag(
    df_categories,
    [
        "belongs_serverless_event_driven",
        "has_serverless_event_driven",
        "has_event_driven"
    ],
    "Serverless Event-Driven"
)

# ---------------------------------------------------------
# 6. Build overlapping category membership
# ---------------------------------------------------------
category_masks = {
    "Edge": edge_flag,
    "HPC": hpc_flag,
    "Serverless": serverless_flag,
    "None": none_flag,
    "Edge no CloudFront": edge_no_cloudfront_flag,
    "Serverless FaaS": faas_flag,
    "Serverless Compute": serverless_compute_flag,
    "Serverless Event-Driven": event_driven_flag
}

category_order = [
    "HPC",
    "Edge",
    "Serverless",
    "None",
    "Edge no CloudFront",
    "Serverless FaaS",
    "Serverless Compute",
    "Serverless Event-Driven"
]

membership_rows = []
category_counts = []

for category in category_order:
    mask = category_masks[category]

    architectures = (
        df_categories.loc[
            mask,
            "architecture"
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    category_counts.append({
        "AnalyticalCategory": category,
        "Architectures": len(architectures)
    })

    for architecture in architectures:
        membership_rows.append({
            "architecture": architecture,
            "AnalyticalCategory": category
        })

df_category_membership = pd.DataFrame(membership_rows)

df_category_counts = pd.DataFrame(category_counts)

print("\nArchitectures by category:")
display(df_category_counts)

# ---------------------------------------------------------
# 7. Merge graph metrics with category memberships
# Each graph can appear in multiple rows because categories overlap.
# ---------------------------------------------------------
df_metrics_by_category = df_metrics.merge(
    df_category_membership,
    on="architecture",
    how="inner"
)

df_metrics_by_category["AnalyticalCategory"] = pd.Categorical(
    df_metrics_by_category["AnalyticalCategory"],
    categories=category_order,
    ordered=True
)

df_metrics_by_category = df_metrics_by_category.sort_values(
    ["AnalyticalCategory", "architecture"]
).reset_index(drop=True)

print("\nGraph metrics rows after category expansion:", len(df_metrics_by_category))
print(
    "Unique graphs represented:",
    df_metrics_by_category["architecture"].nunique()
)

# ---------------------------------------------------------
# 8. Numeric graph metrics for descriptive analysis
# ---------------------------------------------------------
numeric_metrics = [
    "nodes",
    "edges",
    "density",
    "average_degree",
    "average_in_degree",
    "average_out_degree",
    "maximum_path_length",
    "branching_nodes",
    "additional_branches",
    "maximum_branching_factor",
    "source_nodes",
    "sink_nodes",
    "weak_components",
    "strong_components",
    "nodes_with_service_type",
    "number_of_service_types",
    "service_type_entropy",
    "normalized_service_type_entropy"
]

numeric_metrics = [
    column
    for column in numeric_metrics
    if column in df_metrics_by_category.columns
]

# Convert metric columns to numeric safely
for metric in numeric_metrics:
    df_metrics_by_category[metric] = pd.to_numeric(
        df_metrics_by_category[metric],
        errors="coerce"
    )

# ---------------------------------------------------------
# 9. Descriptive summary by category
# ---------------------------------------------------------
summary_rows = []

for category in category_order:
    subset = df_metrics_by_category[
        df_metrics_by_category["AnalyticalCategory"] == category
    ].copy()

    row = {
        "AnalyticalCategory": category,
        "Graphs": subset["architecture"].nunique()
    }

    for metric in numeric_metrics:
        values = subset[metric].dropna()

        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_median"] = values.median()
        row[f"{metric}_std"] = values.std()

    summary_rows.append(row)

df_graph_summary = pd.DataFrame(summary_rows)

# Mean-only compact matrix for easier comparisons
df_graph_means = df_graph_summary[
    [
        "AnalyticalCategory",
        "Graphs"
    ] + [
        f"{metric}_mean"
        for metric in numeric_metrics
    ]
].copy()

# ---------------------------------------------------------
# 10. Export detailed and summary datasets
# ---------------------------------------------------------
df_category_counts.to_csv(
    "/content/graph_metrics_category_counts.csv",
    index=False
)

df_category_membership.to_csv(
    "/content/graph_metrics_category_membership.csv",
    index=False
)

df_metrics_by_category.to_csv(
    "/content/graph_metrics_selected_categories_long.csv",
    index=False
)

df_graph_summary.to_csv(
    "/content/graph_metrics_summary_selected_categories.csv",
    index=False
)

df_graph_means.to_csv(
    "/content/graph_metrics_mean_selected_categories.csv",
    index=False
)

# ---------------------------------------------------------
# 11. Export one graph-metrics CSV per category
# ---------------------------------------------------------
for category in category_order:
    subset = df_metrics_by_category[
        df_metrics_by_category["AnalyticalCategory"] == category
    ].copy()

    output_file = (
        f"/content/graph_metrics_{safe_filename(category)}.csv"
    )

    subset.to_csv(
        output_file,
        index=False
    )

# ---------------------------------------------------------
# 12. Validation and display
# ---------------------------------------------------------
classified_architectures = set(
    df_categories["architecture"].astype(str)
)

metric_architectures = set(
    df_metrics["architecture"].astype(str)
)

missing_metrics = classified_architectures - metric_architectures
missing_classification = metric_architectures - classified_architectures

print("\nValidation:")
print(
    "Architectures in classification but not in graph metrics:",
    len(missing_metrics)
)

print(
    "Graphs without a classification record:",
    len(missing_classification)
)

if missing_metrics:
    print(
        "Example missing graph metrics:",
        sorted(list(missing_metrics))[:10]
    )

if missing_classification:
    print(
        "Example missing classifications:",
        sorted(list(missing_classification))[:10]
    )

print("\nMean graph metrics by category:")
display(df_graph_means.round(4))

print("\nDetailed graph metrics with overlapping category membership:")
display(df_metrics_by_category.head(15))

print("\nFiles exported:")
print("- /content/graph_metrics_category_counts.csv")
print("- /content/graph_metrics_category_membership.csv")
print("- /content/graph_metrics_selected_categories_long.csv")
print("- /content/graph_metrics_summary_selected_categories.csv")
print("- /content/graph_metrics_mean_selected_categories.csv")
print("- /content/graph_metrics_edge.csv")
print("- /content/graph_metrics_hpc.csv")
print("- /content/graph_metrics_serverless.csv")
print("- /content/graph_metrics_edge_no_cloudfront.csv")
print("- /content/graph_metrics_serverless_faas.csv")
print("- /content/graph_metrics_serverless_compute.csv")
print("- /content/graph_metrics_serverless_event_driven.csv")
print("- /content/graph_metrics_none.csv")

Using df_graph_metrics already available in memory.
Graphs in graph metrics: 396
Unique architecture IDs in graph metrics: 396
Classification dataset loaded from: /content/classified_architectures_with_metadata.csv
Architectures in classification dataset: 396

Architectures by category:


,AnalyticalCategory,Architectures
0,HPC,15
1,Edge,106
2,Serverless,352
3,None,40
4,Edge no CloudFront,77
5,Serverless FaaS,224
6,Serverless Compute,29
7,Serverless Event-Driven,64



Graph metrics rows after category expansion: 907
Unique graphs represented: 396

Validation:
Architectures in classification but not in graph metrics: 0
Graphs without a classification record: 0

Mean graph metrics by category:


,AnalyticalCategory,Graphs,nodes_mean,edges_mean,density_mean,average_degree_mean,average_in_degree_mean,average_out_degree_mean,maximum_path_length_mean,branching_nodes_mean,additional_branches_mean,maximum_branching_factor_mean,source_nodes_mean,sink_nodes_mean,weak_components_mean,strong_components_mean,nodes_with_service_type_mean,number_of_service_types_mean,service_type_entropy_mean,normalized_service_type_entropy_mean
0,HPC,15,9.0000,13.1333,0.1906,2.8936,1.4468,1.4468,2.6000,3.4000,5.0000,3.1333,1.3333,0.8667,1.1333,5.0000,9.0000,8.0000,2.9082,0.9806
1,Edge,106,9.6415,12.5943,0.1599,2.5887,1.2944,1.2944,2.9340,3.0943,4.7358,2.9434,1.8019,1.7830,1.8208,5.8491,9.6415,8.4623,2.9711,0.9786
2,Serverless,352,9.1051,11.7273,0.1681,2.5471,1.2736,1.2736,2.7102,2.8409,4.2898,2.7955,1.7102,1.6676,1.7500,5.5256,9.1051,7.7528,2.8364,0.9744
3,None,40,6.1250,7.1000,0.2150,1.8772,0.9386,0.9386,1.6500,1.5000,2.9000,1.7750,1.8000,1.9250,1.8750,4.2750,6.1250,4.8250,2.0029,0.9180
4,Edge no CloudFront,77,9.5584,12.5325,0.1663,2.6151,1.3075,1.3075,3.0130,3.0130,4.6104,2.9091,1.7792,1.6364,1.7532,5.8312,9.5584,8.3247,2.9420,0.9776
5,Serverless FaaS,224,9.4464,12.3973,0.1694,2.6375,1.3188,1.3188,2.8571,3.0402,4.4911,2.8304,1.6339,1.5402,1.6786,5.6071,9.4464,8.0848,2.9090,0.9764
6,Serverless Compute,29,9.8276,12.3103,0.1435,2.4148,1.2074,1.2074,2.6207,2.8966,4.5517,2.3793,2.5172,2.0690,2.2414,6.3448,9.8276,8.5862,2.9831,0.9791
7,Serverless Event-Driven,64,9.1719,11.2188,0.1606,2.4185,1.2092,1.2092,3.0000,2.4219,3.7031,2.6250,1.6562,1.6562,1.7031,5.7812,9.1719,7.8750,2.8632,0.9763



Detailed graph metrics with overlapping category membership:


,file,graph_type,is_directed,nodes,edges,density,average_degree,average_in_degree,average_out_degree,maximum_path_length,...,sink_nodes,weak_components,strong_components,service_type_attribute,nodes_with_service_type,number_of_service_types,service_type_entropy,normalized_service_type_entropy,architecture,AnalyticalCategory
0,2XVgpMwY5iE.graphml,MultiDiGraph,True,7,12,0.285714,3.428571,1.714286,1.714286,1,...,0,1,3,service,7,7,2.807355,1.000000,2XVgpMwY5iE,HPC
1,53sUjFv9ByI.graphml,DiGraph,True,6,6,0.200000,2.000000,1.000000,1.000000,4,...,1,1,6,service,6,6,2.584963,1.000000,53sUjFv9ByI,HPC
2,6CgqEzyWpeA.graphml,DiGraph,True,12,19,0.143939,3.166667,1.583333,1.583333,3,...,0,1,4,service,12,9,3.022055,0.953352,6CgqEzyWpeA,HPC
3,7LziNjUTo7w.graphml,DiGraph,True,9,16,0.222222,3.555556,1.777778,1.777778,1,...,0,1,2,service,9,8,2.947703,0.982568,7LziNjUTo7w,HPC
4,CTG23wd9H74.graphml,DiGraph,True,8,11,0.196429,2.750000,1.375000,1.375000,0,...,0,1,1,service,8,8,3.000000,1.000000,CTG23wd9H74,HPC
5,Dxq_U1TNx1s.graphml,MultiDiGraph,True,11,17,0.154545,3.090909,1.545455,1.545455,4,...,1,1,7,service,11,9,3.095795,0.976615,Dxq_U1TNx1s,HPC
6,EiljRL0977M.graphml,MultiDiGraph,True,7,13,0.309524,3.714286,1.857143,1.857143,3,...,1,1,5,service,7,7,2.807355,1.000000,EiljRL0977M,HPC
7,JRDGId6N49E.graphml,DiGraph,True,6,3,0.100000,1.000000,0.500000,0.500000,1,...,3,3,6,service,6,5,2.251629,0.969724,JRDGId6N49E,HPC
8,SxFag4CMWU8.graphml,DiGraph,True,11,14,0.127273,2.545455,1.272727,1.272727,4,...,1,1,7,service,11,9,3.095795,0.976615,SxFag4CMWU8,HPC
9,U1P8vZTEB-k.graphml,MultiDiGraph,True,12,14,0.106061,2.333333,1.166667,1.166667,5,...,4,1,11,service,12,12,3.584963,1.000000,U1P8vZTEB-k,HPC



Files exported:
- /content/graph_metrics_category_counts.csv
- /content/graph_metrics_category_membership.csv
- /content/graph_metrics_selected_categories_long.csv
- /content/graph_metrics_summary_selected_categories.csv
- /content/graph_metrics_mean_selected_categories.csv
- /content/graph_metrics_edge.csv
- /content/graph_metrics_hpc.csv
- /content/graph_metrics_serverless.csv
- /content/graph_metrics_edge_no_cloudfront.csv
- /content/graph_metrics_serverless_faas.csv
- /content/graph_metrics_serverless_compute.csv
- /content/graph_metrics_serverless_event_driven.csv
- /content/graph_metrics_none.csv
